In [ ]:
from the_well.data.datasets import WellDataset
from pathlib import Path
import sys
import os

PROJECT_ROOT = "/glade/work/gaccarino/waveletflux"
sys.path.insert(0, PROJECT_ROOT)

from waveletflux.utils.mini_well import create_mini_well

In [ ]:
#well_base_path    = '/Users/gabrieleaccarino/Workspace/multi-scale-wavelet-flow-matching/data/datasets'
well_base_path    = '/glade/work/gaccarino/data/datasets'
#well_dataset_name = 'turbulent_radiative_layer_2D'
#well_dataset_name = 'rayleigh_benard'
well_dataset_name = 'planetswe'

MINI_DATASET  = f'{well_dataset_name}_mini'

for split in ['train', 'valid', 'test']:
    ds = WellDataset(
        well_base_path=well_base_path,
        well_dataset_name=well_dataset_name,
        well_split_name=split,
    )
    print(f"\n── {split} ──")
    print(f"  Spatial resolution   : {ds.size_tuple}")
    print(f"  N files              : {ds.n_files}")
    print(f"  Trajectories/file    : {ds.n_trajectories_per_file}")
    print(f"  Steps/traj           : {ds.n_steps_per_trajectory}")
    print(f"  Total trajectories   : {sum(ds.n_trajectories_per_file)}")

In [ ]:
WellDataset(well_base_path=well_base_path,
            well_dataset_name=well_dataset_name,
            well_split_name='test').files_paths

In [ ]:
ds[0].keys()

# Rayleigh-Bénard

In [ ]:
# ── Resampling config ─────────────────────────────────────────
SPATIAL_FACTOR = 1    # no spatial downsampling
TIME_FRACTION  = 1.0  # use all steps per trajectory
TIME_FACTOR    = 1    # no temporal downsampling

MAX_TRAJ_PER_FILE = {'train': 10, 'valid': 5, 'test': 5}

In [ ]:
# ── Parameter subset ─────────────────────────────────────────
ds_full = WellDataset(
    well_base_path=well_base_path,
    well_dataset_name=well_dataset_name,
    well_split_name='train',  # names are identical across splits
)
all_files = sorted(ds_full.files_paths)

KEEP_RA = ['1e6', '1e8', '1e10']   # soft / hard / ultimate turbulence
KEEP_PR = ['1e-1', '1', '10']      # low / mid / high Prandtl

# Anchor Pr match to .hdf5 suffix to avoid '1' matching '10' or '1e-1'
selected_names = [
    Path(f).name for f in all_files
    if any(f'Rayleigh_{ra}_' in Path(f).name for ra in KEEP_RA)
    and any(f'Prandtl_{pr}.hdf5' in Path(f).name for pr in KEEP_PR)
]
print(f"Keeping {len(selected_names)} files:")
for name in selected_names:
    print(f"  {name}")

In [ ]:
# ── Build mini dataset ────────────────────────────────────────
for split, max_traj in MAX_TRAJ_PER_FILE.items():
    print(f"\n{'='*50}  {split}  {'='*50}")
    for fname in selected_names:
        ds_single = WellDataset(
            well_base_path=well_base_path,
            well_dataset_name=well_dataset_name,
            well_split_name=split,
            include_filters=[fname],
        )
        create_mini_well(
            dataset=ds_single,
            well_base_path=well_base_path,
            mini_dataset_name=MINI_DATASET,
            spatial_downsample_factor=SPATIAL_FACTOR,
            time_downsample_factor=TIME_FACTOR,
            max_trajectories=max_traj,
            split=split,
            time_fraction=TIME_FRACTION,
        )

print("\nAll splits done ✓")

# PlanetSWE

In [ ]:
# ── Resampling config ─────────────────────────────────────────
SPATIAL_FACTOR = 1    # no spatial downsampling
TIME_FRACTION  = 1.0  # 1008 × 1.0 ~ 1008 steps, this is 1 model year 
TIME_FACTOR    = 6    # downsample every 6 steps → 168 effective steps
                      # each retained step = 6 sim-hours
                      # daily  cycle: 24/6 =  4 points per day 
                      # annual cycle: 1008/6 = 168 points/year

MAX_TRAJ_PER_FILE = {'train': 1, 'valid': 1, 'test': 1}

In [ ]:
# ── Parameter subset ─────────────────────────────────────────
ds_splits = {
    split: WellDataset(
        well_base_path=well_base_path,
        well_dataset_name=well_dataset_name,
        well_split_name=split,
    )
    for split in ['train', 'valid', 'test']
}

# ── File selection: ALL ICs, only s1 (year 1 out of 3 years) ─────────────────
#   train: IC00–IC31, s1 only → 32 files
#   valid: IC32–IC35, s1 only →  4 files
#   test:  IC36–IC39, s1 only →  4 files

file_selection = {
    split: [
        Path(f).name
        for f in sorted(ds_splits[split].files_paths)
        if Path(f).stem.endswith('_s1')      # first model year only
    ]
    for split in ['train', 'valid', 'test']
}


#KEEP_ICS = [f'IC{i:02d}' for i in range(0, 32, 4)]
#print(f"Keeping train ICs: {KEEP_ICS}")

#file_selection = {
#    'train': [Path(f).name for f in sorted(ds_splits['train'].files_paths) if any(ic in Path(f).name for ic in KEEP_ICS)],
#    'valid': [Path(f).name for f in sorted(ds_splits['valid'].files_paths)],
#    'test' : [Path(f).name for f in sorted(ds_splits['test'].files_paths)],
#}

for split, fnames in file_selection.items():
    print(f"{split}: {len(fnames)} files")

In [ ]:
for split in ['train', 'valid', 'test']:
    print(f"\n{'='*50}  {split}  {'='*50}")
    for fname in file_selection[split]:
        ds_single = WellDataset(
            well_base_path=well_base_path,
            well_dataset_name=well_dataset_name,
            well_split_name=split,
            include_filters=[fname],
        )
        create_mini_well(
            dataset=ds_single,
            well_base_path=well_base_path,
            mini_dataset_name=MINI_DATASET,
            spatial_downsample_factor=SPATIAL_FACTOR,
            time_downsample_factor=TIME_FACTOR,
            max_trajectories=1,
            split=split,
            time_fraction=TIME_FRACTION,
        )

print("\nAll splits done ✓")

# Double check

In [ ]:
# Cell 5 — Verify the mini dataset
for split in ['train', 'valid', 'test']:
    ds_mini = WellDataset(
        path=os.path.join(well_base_path, MINI_DATASET, 'data', split), 
        #well_base_path=well_base_path,
        #well_dataset_name=MINI_DATASET,
        #well_split_name=split,
    )
    total_traj = sum(ds_mini.n_trajectories_per_file)
    print(f"\n── mini/{split} ──")
    print(f"  Spatial resolution : {ds_mini.size_tuple}")
    print(f"  N files            : {ds_mini.n_files}")
    print(f"  Trajectories/file  : {ds_mini.n_trajectories_per_file}")
    print(f"  Steps/traj         : {ds_mini.n_steps_per_trajectory}")
    print(f"  Total trajectories : {total_traj}")
    print(f"  Total windows      : {len(ds_mini)}")